In [ ]:
!pip install feedparser
!pip install pandas
!pip install google-auth
!pip install google-api-python-client
!pip install requests
!pip install openai
!pip install gspread
!pip install oauth2client
!pip install python-dotenv
!pip install jinja2

In [ ]:
import re,os
import feedparser
import hashlib
from datetime import datetime, timezone
import pandas as pd
from google.auth import default
from googleapiclient.discovery import build
from google.oauth2 import service_account
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from dotenv import load_dotenv; load_dotenv()
import requests
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

In [ ]:
# ここでスプレッドシートIDを指定（関数定義は下側の1つだけを使うように重複を削除）
spreadsheet_id = "1WlamXyzIj6GZAkU_lc8C0mTvMzwoHZk-R_HodUC3Sws"
spreadsheet_range = "rss_list!A2:A"

# スプレッドシートからRSSリスト取得する関数
def get_urls_from_spreadsheet():
    # GoogleAPI認証
    # Google APIのスコープを設定
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

    # 認証情報を読み込む
    credentials = ServiceAccountCredentials.from_json_keyfile_name("abiding-ascent-476815-q6-56a05b29f113.json", scope)

    # Googleスプレッドシートに接続（gspreadを使用）
    client = gspread.authorize(credentials)
    # スプレッドシートを開く
    sh = client.open_by_key(spreadsheet_id)
    # 指定範囲の値を取得（A列の2行目以降）
    try:
        values = sh.worksheet('rss_list').get(spreadsheet_range)
    except Exception:
        # フォールバック：列Aの値を取得してヘッダー行をスキップ
        values = [[v] for v in sh.worksheet('rss_list').col_values(1)[1:]]
    flat_urls = [item[0] for item in values if item]
    return flat_urls

# HTMLタグを削除する関数(補助関数)
def clean_html(text):
    text_without_tags = re.sub(r'<[^>]*>', ' ', text)
    cleaned_text = re.sub(r'\s+', ' ', text_without_tags)
    return cleaned_text.strip()

# スプレッドシートから企業名リスト取得する関数
def get_company_names_from_spreadsheet():
    # GoogleAPI認証
    # Google APIのスコープを設定
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

    # 認証情報を読み込む
    credentials = ServiceAccountCredentials.from_json_keyfile_name("abiding-ascent-476815-q6-56a05b29f113.json", scope)

    # Googleスプレッドシートに接続（gspreadを使用）
    client = gspread.authorize(credentials)
    # スプレッドシートを開く
    sh = client.open_by_key(spreadsheet_id)
    # 企業名列の値を取得（B列の2行目以降）
    try:
        values = sh.worksheet('rss_list').get("rss_list!B2:B")
    except Exception:
        # フォールバック：列Bの値を取得してヘッダー行をスキップ
        values = [[v] for v in sh.worksheet('rss_list').col_values(2)[1:]]
    flat_company_names = [item[0] for item in values if item]
    return flat_company_names

# メイン関数
def process_rss_feed():
    # スプレッドシートからRSSリストを取得
    urls = get_urls_from_spreadsheet()
    # スプレッドシートから企業名リストを取得
    company_names = get_company_names_from_spreadsheet()
    tokyo_tz = ZoneInfo('Asia/Tokyo')
    today_jst = datetime.now(tokyo_tz).date()
    
    # RSSフィードからURLリストを取得
    dfs = []
    for i, url in enumerate(urls):
        try:
            f = feedparser.parse(url)
            entries = f.get("entries", [])
            df = pd.json_normalize(entries)
        except:
            continue

        # Ensure expected columns exist to avoid KeyError
        for col in ['title', 'summary', 'link', 'published', 'get_date', 'updated']:
            if col not in df.columns:
                df[col] = ''

        # googleアラート用のURLを削除 (use regex=False to avoid warnings)
        df['link'] = df['link'].astype(str).str.replace('https://www.google.com/url?rct=j&sa=t&url=', '', regex=False)
        
        # 追加：feedparserの parsed 時刻を保持（UTC基準のstruct_time想定）
        # entries と df の順序は対応しているため、同じ順でカラムに入れる
        pub_parsed = [e.get('published_parsed') for e in entries]
        upd_parsed = [e.get('updated_parsed') for e in entries]
        df['published_parsed'] = pub_parsed
        df['updated_parsed']   = upd_parsed
        
        # タイトルと要約をクリーニング
        df['title'] = df['title'].astype(str).apply(clean_html)
        df['summary'] = df['summary'].astype(str).apply(clean_html)

        # Clean published; if missing/empty, fallback to get_date or updated
        df['published'] = df['published'].astype(str).apply(clean_html)
        # If published is empty, try get_date then updated
        df['published'] = df['published'].where(df['published'].str.strip() != '', df['get_date'].astype(str))
        df['published'] = df['published'].where(df['published'].str.strip() != '', df['updated'].astype(str))
        # Final cleanup in case fallback had HTML/extra spaces
        df['published'] = df['published'].astype(str).apply(clean_html)
        
        # 追加：UTC→JSTの日付に変換して“今日のみ”にフィルタ
        def to_jst_date(st):
            # st は time.struct_time 互換（Noneの可能性あり）
            try:
                if st:
                    # feedparserの *_parsed はUTC相当として扱い、JSTへ変換
                    dt_utc = datetime(*st[:6], tzinfo=timezone.utc)
                    return dt_utc.astimezone(tokyo_tz).date()
            except Exception:
                pass
            return None

        df['__pub_date_jst'] = df['published_parsed'].apply(to_jst_date)
        # published_parsed が無い場合は updated_parsed をフォールバック
        missing_mask = df['__pub_date_jst'].isna()
        df.loc[missing_mask, '__pub_date_jst'] = df.loc[missing_mask, 'updated_parsed'].apply(to_jst_date)

        # 今日（JST）のみ残す
        df = df[df['__pub_date_jst'] == today_jst].copy()

        # ヘルパーカラムは捨てる（最終返却スキーマは据え置き）
        df.drop(columns=['published_parsed', 'updated_parsed', '__pub_date_jst'], errors='ignore', inplace=True)

        # 企業名をスプシから追加（各URLのインデックスに対応する企業名を割り当て）
        if i < len(company_names):
            company_name = company_names[i]
        else:
            company_name = ''  # 企業名リストが足りない場合は空文字
        
        df['company_name'] = company_name
        dfs.append(df)

    if not dfs:
        # Return empty dataframe with expected columns
        return pd.DataFrame(columns=['title', 'summary', 'link', 'published', 'company_name'])

    df = pd.concat(dfs, ignore_index=True)

    # 必要なカラムだけ抽出
    # Make sure columns exist before selecting
    for col in ['title', 'summary', 'link', 'published', 'company_name']:
        if col not in df.columns:
            df[col] = ''
    df = df[['title', 'summary', 'link', 'published', 'company_name']]
    return df

# 実行
df_rss_list = process_rss_feed()

In [ ]:
df_rss_list.head(10)

In [ ]:
df_display = df_rss_list.copy()

# 改行や余計な空白を整理
df_display["title"] = df_display["title"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df_display["link"] = df_display["link"].astype(str).str.strip()

from IPython.display import display
pd.set_option("display.max_colwidth", None)

# Discord送信用（タイトルとリンクをまとめたテキスト）
entries = [
    f"{row['title']}\n{row['link']}"
    for _, row in df_display.iterrows()
]
text = "\n\n".join(entries)

# 環境変数がセットされていればそれを使い、なければ既存の変数を利用
WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")
r = requests.post(WEBHOOK_URL, json={"content": text})